## HPO for XOR3 using Binary Genetic Algorithm (BGA)

### BGA code based on:
https://youtu.be/EJeTWRP3Bd0
### 3 classes

<pre>
   | x0 | x1 |XOR3|
   |----|----|----|
   |0.0 |0.0 | 0  |
   |0.0 |1.0 | 1  |
   |1.0 |0.0 | 1  |
   |1.0 |1.0 | 0  |
   |0.5 |0.5 | 2  |
</pre>

Hyper parameters to optimize
- number of hidden neurons
- learning rate
- batch size
- num epochs
- activation function
- optimizer algorithm

0: num_neurons, 1: lr, 2: bsize, 3: mum epochs, 4: actfun, 5: optmzr


## Objective Function Definition Using Keras

In [1511]:
# The objective function is a two-dimensional inverted Gaussian function, centred at (7, 9)
# def objective(x):
#   return math.exp(((x[0]-7)**2) + (x[1]-9)**2)
#   return (1-x[0])**2 + (x[1]-x[0]**2)**2
#   return x[0]**2 + 25*x[1]**2

In [1512]:
act_func = ('relu', 'elu', 'sigmoid', 'tanh', 'leaky_relu') # order does not matter
optimz = ('SGD', 'RMSprop', 'Adam') # order does not matter
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation
from keras import optimizers
from keras.models import save_model, load_model
import matplotlib.pyplot as plt
import numpy as np
import keras

# the five different states of the XOR3 gate
X = np.array([[0,0],[0,1],[1,0],[1,1],[0.5,0.5]])

# the five expected results in the same order
y = np.array([[0],[1],[1],[0],[2]])

Xt = np.array([[0,0.1],[0.1,1],[0.9,0],[0.9,1],[0.55,0.45]])
yt = np.array([[0],[1],[1],[0],[2]])

# 0: num_neurons,    1: lr,    2: bsize,    3: mum epochs,  4: act_fn,  5: optmzr

def objective(param): # XOR3 evaluation function
  global model

  model = Sequential([
    keras.Input(shape=(2,)),
    Dense(int(param[0]), activation=act_func[ int(param[4]) ]),
    Dense(3, activation='softmax')
  ])
  if ( int(param[5]) == 0 ):
    optmzr = optimizers.SGD(learning_rate=param[1]) # stochastic gradient decent
  elif ( int(param[5]) == 1 ):
    optmzr = optimizers.RMSprop(learning_rate=param[1])
  elif ( int(param[5]) == 2 ):
    optmzr = optimizers.Adam(learning_rate=param[1])

  model.compile(loss='sparse_categorical_crossentropy', optimizer=optmzr, metrics=['accuracy'])
  model.fit(X, y, batch_size=int(param[2]), epochs=int(param[3]), verbose=0)
  (loss, acc) = model.evaluate(Xt, yt, verbose=0)

  return loss

## BGA

In [1513]:
from numpy.random import randint
from numpy.random import rand
import math

In [1514]:
# The decode function decodes binary bitstrings to numbers for each input and scales the value to fall within defined bounds
def decode(bounds, n_bits, bitstring):
	"""
	This function decodes binary bitstrings to numbers for each input and scales the value to fall within defined bounds.

	Parameters:
		bounds (list): A list of tuples representing the lower and upper bounds for each value to be decoded. [lower, upper)
		n_bits (int): The number of bits used to represent each value.
		bitstring (list): A binary bitstring to be decoded.

	Returns:
		decoded (list): A list of decoded values.
	"""
	decoded = []  # Create an empty list to hold the decoded values
	largest = 2**n_bits  # The largest value that can be represented with the given number of bits
	for i in range(len(bounds)):
		# Extract the substring for the current value
		start, end = i * n_bits, (i * n_bits) + n_bits  # Define the start and end indices of the substring
		substring = bitstring[start:end]  # Extract the substring
		# Convert the substring to a string of characters
		chars = ''.join([str(s) for s in substring])  # Join the values in the substring together into a string of characters
		# Convert the string of characters to an integer
		integer = int(chars, 2)  # Convert the binary number string into an integer
		# Scale the integer to the desired range
		value = bounds[i][0] + (integer/largest) * (bounds[i][1] - bounds[i][0])  # Scale the integer to a value within the defined bounds
		# Store the decoded value
		decoded.append(value)
	return decoded

In [1515]:
def selection(pop, scores, k=3):
    """
    Select the best individuals for the next generation based on their fitness (scores).
    This function randomly selects k individuals from the population and performs a tournament
    among them to choose the one with the best score.

    Parameters:
    pop (list): The population of individuals.
    scores (list): The fitness scores for each individual in the population.
    k (int, optional): The number of individuals to select from the population for the tournament.
                        Defaults to 3.

    Returns:
    individual: The best individual from the tournament.
    """
    # Randomly select one index from the population as the initial selection
    selection_ix = randint(len(pop))
    # Perform a tournament among k randomly selected individuals
    for ix in randint(0, len(pop), k-1):
        # Check if the current individual has a better score than the selected one
        if scores[ix] < scores[selection_ix]:
            # Update the selected individual if a better one is found
            selection_ix = ix
    # Return the best individual from the tournament
    return pop[selection_ix]

In [1516]:
def crossover(p1, p2, r_cross):
    """
    Create two children from two parents using the crossover operation.
    The children are created by copying the parents, and recombination occurs
    if a random value is less than the crossover rate.

    Parameters:
    p1 (list): The first parent.
    p2 (list): The second parent.
    r_cross (float): The crossover rate.

    Returns:
    list: A list containing the two children.
    """
    # Children are copies of the parents by default
    c1, c2 = p1.copy(), p2.copy()
    # Check if recombination should occur
    if rand() < r_cross:
        # Select a crossover point (not at the end of the string)
        pt = randint(1, len(p1)-2)
        # Perform crossover in the children
        c1 = p1[:pt] + p2[pt:]
        c2 = p2[:pt] + p1[pt:]
    # Return the two children
    return [c1, c2]

In [1517]:
#The crossover process can generate offsprings that are very similar to the parents. This might cause a new generation with low diversity.
# The mutation process solves this problem by changing the value of some features in the offspring at random.

import random

def mutation(bitstring, r_mut):
    """
    The mutation process changes the value of some features in the offspring at random to maintain the diversity in the population.
    A standard value for the mutation rate is 1/m where m is the number of features.

    Parameters:
    bitstring (list): A list of binary values representing the offspring
    r_mut (float): The mutation rate, typically 1/m where m is the number of features

    Returns:
    list: The modified bitstring after mutation

    """
    rand = random.random
    for i in range(len(bitstring)):
        # Check for a mutation
        if rand() < r_mut:
            # Flip the bit
            bitstring[i] = 1 - bitstring[i]
    return bitstring

In [1518]:
### Putting all together into our Genetic algorithm that runs until it finds the best
#The whole fitness assignment, selection, recombination, and mutation process is
#repeated until a stopping criterion is satisfied.
#Each generation is likely to be more adapted to the environment than the old one.

# genetic algorithm implementation
def genetic_algorithm(objective, bounds, n_bits, n_iter, n_pop, r_cross, r_mut):
    """
    The genetic algorithm that finds the optimal solution by performing the fitness assignment, selection, recombination, and mutation process repeatedly.
    Each iteration, the solution is more adapted to the environment.

    Parameters
    ----------
    objective: function
        The objective function that needs to be optimized.
    bounds: list of tuples
        The bounds of the solution.
    n_bits: int
        The number of bits used to encode the solution.
    n_iter: int
        The number of iterations to perform.
    n_pop: int
        The size of the population.
    r_cross: float
        The crossover rate.
    r_mut: float
        The mutation rate.

    Returns
    -------
    list
        The best solution and its evaluation.
    """
    # initialize the population with random bitstrings
    pop = [randint(0, 2, n_bits * len(bounds)).tolist() for _ in range(n_pop)]

    # track the best solution found so far
    best, best_eval = 0, objective(decode(bounds, n_bits, pop[0]))

    # iterate over generations
    for gen in range(n_iter):
        # decode the population
        decoded = [decode(bounds, n_bits, p) for p in pop]
        # evaluate all candidates in the population
        scores = [objective(d) for d in decoded]
        # check for a new best solution
        for i in range(n_pop):
            if scores[i] < best_eval:
                best, best_eval = pop[i], scores[i]
                print(f"Gen={gen}, new best f({decoded[i]}) = {scores[i]}")
                if scores[i] < 0.005:
                    print(f"termination condition met at gen={gen}")
                    return [best, best_eval]

        # select parents
        selected = [selection(pop, scores) for _ in range(n_pop)]

        # create the next generation - children
        children = list()
        for i in range(0, n_pop, 2):
            # get selected parents in pairs
            p1, p2 = selected[i], selected[i + 1]
            # crossover and mutation
            for c in crossover(p1, p2, r_cross):
                # mutation
                mutation(c, r_mut)
                # store for next generation
                children.append(c)
        # replace the population
        pop = children
    return [best, best_eval]


In [1519]:
def print_HP_found(x, eval):
  print(f"#neurons={int(x[0])}, lr={x[1]:.3f}, bsize={int(x[2])}, #epochs={int(x[3])}, actF={act_func[int(x[4])]},  optim={optimz[int(x[5])]}")
  print(f"Eval={eval}")
#################### M A I N #########################
# bounds = [[-2, 2], [-2, 2]]
# range for input variables (hyper params)
bounds = [[2.0, 17.0], # num of hidden neurons [low, high) ==> int
          [0.01, 1.5], # lr
          [1.0, 5.0], # batch size ==> int
          [100.0, 500.0], # num epochs ==> int
          [0.0, 5.0], # activation function out of 5 ==> int
          [0.0, 3.0], # optimizer out of 3 ==> int
         ]

n_iter = 20 # 300 # define the total iterations
n_bits = 8 # 16 # # bits per variable
n_pop = 24 #100 # the population size, **must be an even number**

r_cross = 0.9 # crossover rate
r_mut = 1.0 / (float(n_bits) * len(bounds)) # mutation rate
# perform the genetic algorithm search
best, score = genetic_algorithm(objective, bounds, n_bits, n_iter, n_pop, r_cross, r_mut)
decoded = decode(bounds, n_bits, best)
print('=============')
print_HP_found(decoded, score)

Gen=0, new best f([13.015625, 0.906328125, 4.859375, 285.9375, 0.625, 0.328125]) = 0.39204615354537964
Gen=0, new best f([14.01171875, 1.0169140625, 2.75, 184.375, 4.609375, 2.765625]) = 0.0
termination condition met at gen=0
#neurons=14, lr=1.017, bsize=2, #epochs=184, actF=leaky_relu,  optim=Adam
Eval=0.0
